# GPU XGBoost (Dask) + Optuna multi-objective hyper-parameter tuning

Regression with a positive, right-skewed target that has some extreme values.

* **Data**: `dask_cudf` dataframes → `DaskQuantileDMatrix` (validation matrix built with `ref=dtrain`).
* **Model**: `xgboost.dask.train` on a `LocalCUDACluster` (one worker per GPU).
* **Losses**: MSE (`reg:squarederror`), pseudo-Huber (`reg:pseudohubererror`), Tweedie (`reg:tweedie`, variance power tuned).
* **Eval metrics**: `rmse`, `mae`, `mphe` (+ `tweedie-nloglik@p` for Tweedie). The loss metric is placed last and early stopping (50 rounds) monitors it on the validation set.
* **Tuning**: Optuna `TPESampler(n_startup_trials, gamma)` with two objectives to minimise:
  1. the loss metric on the validation set, and
  2. overfit index = `|valid_metric - train_metric| / train_metric`.
* **Resumable**: every trial is committed to a SQLite study, with its booster + metrics on disk. If the run dies (GPU error, kernel crash), just re-run the tuning cell.

In [ ]:
import functools
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, if the package is not pip-installed

import numpy as np
import pandas as pd

from xgb_optuna_gpu import (
    XGBOptunaTuner, load_dask_data, make_client, make_synthetic_data, train_xgb_model,
    build_dmatrices, get_loss,
)

## 1. Configuration

In [ ]:
USE_GPU = True            # False -> CPU LocalCluster (smoke test without a GPU)

# Your data. Leave as None to generate a synthetic right-skewed dataset for the demo.
TRAIN_PATH = None         # e.g. "s3://bucket/train/*.parquet" or "data/train.parquet"
VALID_PATH = None
TARGET = "y"
FEATURES = None           # None -> every column except TARGET
FILE_FORMAT = "parquet"

N_TRIALS = 50             # max Bayesian optimisation rounds (total, across resumes)
N_STARTUP_TRIALS = 15     # random trials before TPE kicks in
GAMMA_FRACTION = 0.10     # TPE gamma(n) = min(ceil(GAMMA_FRACTION * n), GAMMA_MAX)
GAMMA_MAX = 25
NUM_BOOST_ROUND = 5000    # upper bound; early stopping (50 rounds) decides the real number
EARLY_STOPPING_ROUNDS = 50
MAX_OVERFIT = 0.10        # best model = lowest loss on the Pareto front with overfit <= this
OUTPUT_DIR = Path("optuna_runs")

## 2. Start the Dask (CUDA) cluster

In [ ]:
client = make_client(use_gpu=USE_GPU)   # e.g. make_client(True, rmm_pool_size="20GB")
client

## 3. Data

`data_loader` is a zero-argument function returning `(X_train, y_train, X_valid, y_valid)` as persisted Dask (cuDF)
collections. The tuner calls it lazily — and again if it has to restart the workers after a GPU error.

In [ ]:
if TRAIN_PATH is None:
    data_dir = Path("demo_data"); data_dir.mkdir(exist_ok=True)
    train_df, valid_df = make_synthetic_data(n_train=500_000, n_valid=100_000, n_features=20)
    # Write several files per split so partitions spread over all GPUs.
    import dask.dataframe as dd
    dd.from_pandas(train_df, npartitions=16).to_parquet(data_dir / "train", overwrite=True)
    dd.from_pandas(valid_df, npartitions=8).to_parquet(data_dir / "valid", overwrite=True)
    TRAIN_PATH, VALID_PATH = str(data_dir / "train"), str(data_dir / "valid")

data_loader = functools.partial(
    load_dask_data, TRAIN_PATH, VALID_PATH, TARGET,
    features=FEATURES, fmt=FILE_FORMAT, use_gpu=USE_GPU, client=client,
)
X_train, y_train, X_valid, y_valid = data_loader()
print(f"train: {len(y_train):,} rows x {X_train.shape[1]} features | valid: {len(y_valid):,} rows")

In [ ]:
# Target distribution: positive, right skewed, heavy tail
q = y_train.quantile([0.0, 0.25, 0.5, 0.75, 0.9, 0.99, 0.999, 1.0]).compute()
q = q.to_pandas() if hasattr(q, "to_pandas") else q
mean, std = float(y_train.mean().compute()), float(y_train.std().compute())
print(f"mean={mean:.3f}  std={std:.3f}  min>0: {float(y_train.min().compute()) > 0}")
q.rename("quantile").to_frame()

## 4. A single model (sanity check)

`train_xgb_model` can be used on its own, e.g. to refit a final model. It returns the booster (already truncated to the
best iteration) and a `TrainResult` with eval metrics at the best iteration, average target and average prediction.

In [ ]:
dtrain, dvalid = build_dmatrices(client, X_train, y_train, X_valid, y_valid, max_bin=256)
booster, res = train_xgb_model(
    client, dtrain, dvalid,
    params={"max_depth": 6, "learning_rate": 0.1, "tweedie_variance_power": 1.5, "max_bin": 256},
    loss=get_loss("tweedie"),
    X_train=X_train, y_train=y_train, X_valid=X_valid, y_valid=y_valid,
    num_boost_round=NUM_BOOST_ROUND, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    use_gpu=USE_GPU, save_dir=OUTPUT_DIR / "baseline_tweedie",
)
pd.Series(res.summary()).drop("params")

In [ ]:
del dtrain, dvalid, booster  # free GPU memory; the tuner builds its own DMatrices

## 5. Hyper-parameter tuning

One study per loss function (their loss metrics are on different scales, so they are not comparable within one study).
Tuned: `max_depth, learning_rate, gamma, min_child_weight, max_delta_step, reg_alpha, reg_lambda`
(+ `tweedie_variance_power` for Tweedie). Ranges live in `xgb_optuna_gpu.tuner.DEFAULT_SEARCH_SPACE`; override
with `search_space={name: (kind, low, high, {"log": True})}`.

**Resuming**: re-running a cell continues the same study (same `OUTPUT_DIR` + `study_name`) until `N_TRIALS` complete
trials exist. Trials interrupted by a crash are retried with the same hyper-parameters.

In [ ]:
common = dict(
    client=client, data_loader=data_loader, output_dir=OUTPUT_DIR,
    num_boost_round=NUM_BOOST_ROUND, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    eval_metrics=("rmse", "mae", "mphe"),
    n_startup_trials=N_STARTUP_TRIALS, gamma_fraction=GAMMA_FRACTION, gamma_max=GAMMA_MAX,
    max_overfit=MAX_OVERFIT, use_gpu=USE_GPU, seed=42,
    fixed_params={"subsample": 0.8, "colsample_bytree": 0.8, "max_bin": 256},
)
tuners = {}

### 5a. MSE (`reg:squarederror`, metric `rmse`)

In [ ]:
tuners["mse"] = XGBOptunaTuner(loss="mse", study_name="xgb_mse", **common)
best_mse = tuners["mse"].optimize(n_trials=N_TRIALS)

### 5b. Pseudo-Huber (`reg:pseudohubererror`, metric `mphe`)

`huber_slope` (δ) is in target units: residuals larger than δ are penalised linearly. A high quantile of
`|y - median(y)|` is a reasonable starting point; it can also be added to the search space.

In [ ]:
med = float(y_train.quantile(0.5).compute())
huber_slope = float((y_train - med).abs().quantile(0.9).compute())
print(f"huber_slope = {huber_slope:.4f}")

huber_common = {**common, "fixed_params": {**common["fixed_params"], "huber_slope": huber_slope}}
tuners["huber"] = XGBOptunaTuner(loss="huber", study_name="xgb_huber", **huber_common)
best_huber = tuners["huber"].optimize(n_trials=N_TRIALS)

### 5c. Tweedie (`reg:tweedie`, metric `tweedie-nloglik@p`, `p` tuned in [1.05, 1.95])

Note: `tweedie-nloglik@p` values for different `p` are different likelihoods, so comparing trials with different `p` on
that metric is not strictly apples-to-apples. If that matters, pass `opt_metric="rmse"` (or `"mae"`) — both objectives
(loss and overfit index) then use that metric while the model still trains/early-stops on the Tweedie loss.

In [ ]:
tuners["tweedie"] = XGBOptunaTuner(loss="tweedie", study_name="xgb_tweedie", **common)
best_tweedie = tuners["tweedie"].optimize(n_trials=N_TRIALS)

## 6. Compare the best model of each loss

Validation metrics at each model's best iteration, plus calibration (`avg_pred / avg_target`, ideally ≈ 1).
Compare losses on `rmse` / `mae`: `mphe` is computed with each model's `huber_slope` (1.0 unless set), and each
`valid_loss` is on its own scale.

In [ ]:
rows = {}
for name, t in tuners.items():
    r = t.best_result()
    rows[name] = {
        "trial": t.best_trial().number, "loss_metric": r.loss_metric,
        "valid_loss": r.valid_metrics[r.loss_metric], "overfit_index": r.overfit_index,
        "valid_rmse": r.valid_metrics["rmse"], "valid_mae": r.valid_metrics["mae"],
        "valid_mphe": r.valid_metrics["mphe"], "avg_target_valid": r.avg_target_valid,
        "avg_pred_valid": r.avg_pred_valid, "calibration": r.avg_pred_valid / r.avg_target_valid,
        "best_iteration": r.best_iteration,
    }
comparison = pd.DataFrame(rows).T
comparison

## 7. Pareto front & trial history

In [ ]:
import matplotlib.pyplot as plt
import optuna

fig, axes = plt.subplots(1, len(tuners), figsize=(5 * len(tuners), 4))
for ax, (name, t) in zip(np.atleast_1d(axes), tuners.items()):
    done = np.array([tr.values for tr in t.study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))])
    pf = t.pareto_front()
    best = t.best_trial()
    ax.scatter(done[:, 0], done[:, 1], s=14, alpha=0.4, label="trials")
    ax.plot(pf.loss, pf.overfit_index, "o-", color="C1", label="Pareto front")
    ax.scatter([best.values[0]], [best.values[1]], marker="*", s=250, color="C3", label="selected", zorder=5)
    ax.axhline(MAX_OVERFIT, ls="--", lw=1, color="grey")
    ax.set(title=name, xlabel=t.best_result().loss_metric + " (valid)", ylabel="overfit index")
    ax.legend()
plt.tight_layout()

In [ ]:
tuners["tweedie"].pareto_front().head(10)

In [ ]:
# Full trial log (also saved after every trial to optuna_runs/<study>/trials.csv)
pd.read_csv(tuners["tweedie"].study_dir / "trials.csv").tail()

## 8. Resuming after a GPU failure

Everything needed to resume is on disk:

```
optuna_runs/xgb_tweedie/
├── study.db                 # Optuna SQLite storage, committed after every trial
├── trials.csv               # human-readable snapshot, refreshed after every trial
└── trials/trial_00012/
    ├── booster.ubj          # booster truncated at the best iteration
    └── metrics.json         # params, eval metrics (train/valid), avg target/prediction, full history
```

* **Error inside a trial** (e.g. CUDA OOM): `optimize` restarts the Dask workers, reloads the data, re-queues that
  trial's hyper-parameters and continues (up to `max_restarts`).
* **Kernel / process died**: restart the kernel, run the setup cells and the same `optimize` cell again. Completed trials
  are kept, the interrupted trial is retried, and only the remaining trials up to `N_TRIALS` are run.
* **More rounds later**: call `optimize(n_trials=N_TRIALS + 50)` on the same study.

You can also inspect a study without a GPU:

In [ ]:
study = optuna.load_study(study_name="xgb_tweedie", storage=tuners["tweedie"].storage_url)
print(len(study.trials), "trials;", [t.number for t in study.best_trials], "on the Pareto front")
tuners["tweedie"].report_best(max_overfit=0.05);   # re-select with a stricter overfit threshold

## 9. Load the best booster and predict

In [ ]:
from xgboost import dask as dxgb

best_booster = tuners["tweedie"].load_best_booster()
pred_valid = dxgb.inplace_predict(client, best_booster, X_valid)
print("avg prediction:", float(pred_valid.mean().compute()), "| avg target:", float(y_valid.mean().compute()))

In [ ]:
client.close()